In [6]:
def extract_weights(sid_list, target_len):
    return sid_list[-target_len:]

def compute_similarity(seq1, seq2, weight_seq):
    length = max(len(seq1), len(seq2))
    penalty = 0
    for idx in range(length):
        ch1 = ord(seq1[idx]) if idx < len(seq1) else 0
        ch2 = ord(seq2[idx]) if idx < len(seq2) else 0
        weight = weight_seq[idx] if idx < len(weight_seq) else 1
        penalty += weight * abs(ch1 - ch2)
    return -penalty

def compute_boosted_score(seq1, seq2, weight_seq, boost_start_idx, multiplier):
    length = max(len(seq1), len(seq2))
    penalty = 0
    for idx in range(length):
        ch1 = ord(seq1[idx]) if idx < len(seq1) else 0
        ch2 = ord(seq2[idx]) if idx < len(seq2) else 0
        weight = weight_seq[idx] if idx < len(weight_seq) else 1
        if idx >= boost_start_idx:
            weight *= multiplier
        penalty += weight * abs(ch1 - ch2)
    return -penalty

def gene_search(options, assembled, is_maximizer, alpha, beta, ref_seq, weight_seq,
                allow_boost=False, boost_value=1.0, boost_index=None):
    if not options:
        if allow_boost and boost_index is not None:
            return compute_boosted_score(assembled, ref_seq, weight_seq, boost_index, boost_value), assembled
        return compute_similarity(assembled, ref_seq, weight_seq), assembled

    if is_maximizer:
        max_score = float('-inf')
        best_seq = ""
        for i in range(len(options)):
            pick = options[i]
            remaining = options[:i] + options[i+1:]
            new_seq = assembled + pick
            next_boost_idx = boost_index
            if allow_boost and pick == 'S' and boost_index is None:
                next_boost_idx = len(assembled)
            score, result = gene_search(remaining, new_seq, False, alpha, beta, ref_seq, weight_seq,
                                         allow_boost, boost_value, next_boost_idx)
            if score > max_score:
                max_score = score
                best_seq = result
            alpha = max(alpha, score)
            if beta <= alpha:
                break
        return max_score, best_seq
    else:
        min_score = float('inf')
        best_seq = ""
        for i in range(len(options)):
            pick = options[i]
            remaining = options[:i] + options[i+1:]
            new_seq = assembled + pick
            score, result = gene_search(remaining, new_seq, True, alpha, beta, ref_seq, weight_seq,
                                         allow_boost, boost_value, boost_index)
            if score < min_score:
                min_score = score
                best_seq = result
            beta = min(beta, score)
            if beta <= alpha:
                break
        return min_score, best_seq

def execute_game(nucleotides_str, ref_gene_str, sid_vals):
    elements = nucleotides_str.strip().split(',')
    ref_seq = ref_gene_str.strip()
    weight_sequence = extract_weights(sid_vals, len(ref_seq))
    multiplier = int(f"{sid_vals[0]}{sid_vals[1]}") / 100

    # Task 1
    score_plain, result_plain = gene_search(elements, "", True, float('-inf'), float('inf'), ref_seq, weight_sequence)
    print(f"Best gene sequence generated: {result_plain}")
    print(f"Utility score: {score_plain}")

    # Task 2
    print()
    if 'S' in elements:
        score_boosted, result_boosted = gene_search(elements, "", True, float('-inf'), float('inf'),
                                                    ref_seq, weight_sequence, allow_boost=True,
                                                    boost_value=multiplier)
        if score_boosted > score_plain:
            print("YES")
        else:
            print("NO")
        print("With special nucleotide")
        print(f"Best gene sequence generated: {result_boosted}")
        print(f"Utility score: {round(score_boosted, 2)}")
    else:
        print("No")

if __name__ == "__main__":
    nucleotides_input = "A,T,C,G,S"
    target_gene = "ATGC"
    sid = [2, 1, 3, 0, 1, 2, 5, 9]  # Student ID: 21301259

    execute_game(nucleotides_input, target_gene, sid)


Best gene sequence generated: TASGC
Utility score: -220

YES
With special nucleotide
Best gene sequence generated: SATGC
Utility score: -47.04
